<a href="https://colab.research.google.com/github/mitalidaduria/nlp-payments-lab/blob/main/MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q mlflow xgboost optuna scikit-learn pandas joblib pyngrok

In [2]:
import os

os.makedirs("scripts", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
print("Project directories created successfully!")

Project directories created successfully!


In [12]:
%%writefile scripts/train_with_tracking.py
import os
import joblib
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import mlflow
import mlflow.xgboost

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, auc, f1_score

# Silence Optuna logs for clean execution output
optuna.logging.set_verbosity(optuna.logging.WARNING)

def create_synthetic_data(n_samples=10000, fraud_rate=0.02, n_features=20, random_state=42):
    """Generates synthetic payment fraud data."""
    weights = [1 - fraud_rate, fraud_rate]
    X_raw, y_raw = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=10,
        n_redundant=5,
        weights=weights,
        random_state=random_state
    )
    feature_names = [f"feature_{i}" for i in range(n_features)]
    df = pd.DataFrame(X_raw, columns=feature_names)
    df['is_fraud'] = y_raw
    return df

def main():
    # -------------------------------------------------------------
    # 1. Configuration & MLflow Setup
    # -------------------------------------------------------------
    DATA_CONFIG = {
        "n_samples": 10000,
        "fraud_rate": 0.02,
        "n_features": 20,
        "random_state": 42
    }

    mlflow.set_experiment("payment-fraud-detection")

    with mlflow.start_run(run_name="xgb_optuna_50trials") as run:
        print(f"Started MLflow Run ID: {run.info.run_id}")

        # Log Data Configuration Parameters
        mlflow.log_params({f"data_{k}": v for k, v in DATA_CONFIG.items()})

        # -------------------------------------------------------------
        # 2. Data Preparation & Feature Pipeline Artifact
        # -------------------------------------------------------------
        df = create_synthetic_data(**DATA_CONFIG)
        X = df.drop(columns=["is_fraud"])
        y = df["is_fraud"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=DATA_CONFIG["random_state"]
        )

        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Save feature pipeline artifact locally
        os.makedirs("artifacts", exist_ok=True)
        pipeline_path = "artifacts/feature_pipeline.pkl"
        joblib.dump(scaler, pipeline_path)

        # Log feature pipeline to MLflow
        mlflow.log_artifact(pipeline_path, artifact_path="pipeline")

        # -------------------------------------------------------------
        # 3. Hyperparameter Optimization with Optuna
        # -------------------------------------------------------------
        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 200),
                'max_depth': trial.suggest_int('max_depth', 3, 9),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'scale_pos_weight': (1 - DATA_CONFIG["fraud_rate"]) / DATA_CONFIG["fraud_rate"],
                'random_state': DATA_CONFIG["random_state"],
                'eval_metric': 'aucpr'
            }

            cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=DATA_CONFIG["random_state"])
            pr_aucs = []

            for train_idx, val_idx in cv.split(X_train_scaled, y_train):
                X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
                y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

                model = xgb.XGBClassifier(**params)
                model.fit(X_tr, y_tr)

                preds_probs = model.predict_proba(X_val)[:, 1]
                precision, recall, _ = precision_recall_curve(y_val, preds_probs)
                pr_aucs.append(auc(recall, precision))

            return np.mean(pr_aucs)

        print("Running Optuna study (50 trials)...")
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=50)

        best_params = study.best_params
        print(f"Best Trial PR-AUC: {study.best_value:.4f}")

        # Log Optuna Best Hyperparameters
        mlflow.log_params({f"optuna_{k}": v for k, v in best_params.items()})
        mlflow.log_metric("optuna_best_cv_pr_auc", study.best_value)

        # -------------------------------------------------------------
        # 4. Final Model Training & Evaluation
        # -------------------------------------------------------------
        final_params = {
            **best_params,
            'scale_pos_weight': (1 - DATA_CONFIG["fraud_rate"]) / DATA_CONFIG["fraud_rate"],
            'random_state': DATA_CONFIG["random_state"]
        }

        final_model = xgb.XGBClassifier(**final_params)
        final_model.fit(X_train_scaled, y_train)

        # Evaluate on Test Set
        test_probs = final_model.predict_proba(X_test_scaled)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test, test_probs)
        test_pr_auc = auc(recall, precision)

        test_preds = (test_probs >= 0.5).astype(int)
        test_f1 = f1_score(y_test, test_preds)

        # Log Final Metrics
        mlflow.log_metrics({
            "test_pr_auc": test_pr_auc,
            "test_f1": test_f1
        })

        # Log XGBoost Model Artifact
        mlflow.xgboost.log_model(final_model, artifact_path="xgb_model")

        print(f"Run complete! Test PR-AUC: {test_pr_auc:.4f}, Test F1: {test_f1:.4f}")

if __name__ == "__main__":
    main()

Overwriting scripts/train_with_tracking.py


In [13]:
!python scripts/train_with_tracking.py

Started MLflow Run ID: 0e2f7d2f66c444d5a990a9eb11970d21
Running Optuna study (50 trials)...
Best Trial PR-AUC: 0.5104
2026/08/04 14:55:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Run complete! Test PR-AUC: 0.6054, Test F1: 0.6000


In [17]:
from google.colab import output
import time

# 1. Start MLflow server
get_ipython().system_raw("MLFLOW_SERVER_ALLOWED_HOSTS='*' mlflow server --host 0.0.0.0 --port 5000 &")

time.sleep(3)

# 2. Open MLflow in a native Colab window
output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [18]:
import os
import time
from google.colab import output

# 1. Kill ALL stuck background MLflow servers freeing up port 5000
os.system("pkill -f mlflow")
time.sleep(2)

# 2. Start a fresh, clean MLflow UI server
get_ipython().system_raw("mlflow ui --port 5000 --host 0.0.0.0 &")
time.sleep(3)

# 3. Open the native Colab window again
output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [19]:
import time
from google.colab import output

# 1. Start a fresh MLflow server on a BRAND NEW PORT (5050)
get_ipython().system_raw("MLFLOW_SERVER_ALLOWED_HOSTS='*' mlflow server --host 0.0.0.0 --port 5050 &")

# Give it a few seconds to boot up safely
time.sleep(4)

# 2. Open the native Colab window on the new port
output.serve_kernel_port_as_window(5050)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>